# Etapa 2. Indicadores

**Entrada:** a base consolidada da etapa 1 (`01_conferencia_e_consolidacao/base_consolidada`) e, só para o teste de regressão, os dados brutos e os indicadores da Versão 2 (`versoes_anteriores/Versao_2/arquivos_tabulares`).

**Processo:**
1. Teste de regressão: com as regras da Versão 2 e os dados brutos da Versão 2, o código precisa reproduzir os 20 indicadores da Versão 2 (`src/indicadores.py`).
2. Tabelas dos quatro desenhos, com os 23 indicadores e a marcação de quem entra na análise principal (`src/desenhos.py`).
3. Verificações simples do painel (`src/verificacoes.py`).
4. Resumo estatístico dos indicadores, que alimenta a etapa 3 (`src/resumo.py`).

**Saída** (nesta pasta):
- `tabelas/desenho_A.parquet` a `tabelas/desenho_D.parquet`: tabelas dos desenhos, com todos os registros e a coluna `motivo_fora`.
- `tabelas/verificacoes.parquet`: resultado das verificações por universidade e ano.
- `tabelas_desenhos.xlsx`: cópia das tabelas dos desenhos para consulta no Excel.
- `resumo_indicadores.xlsx`: estatísticas de cada indicador em cada desenho, pares mais correlacionados e alertas das verificações.

Os textos que explicam cada indicador estão em `dicionario_indicadores.xlsx`, editado à mão (D49).

## Preparação

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

# O notebook roda dentro da pasta da etapa. Aqui ele localiza a raiz do repositório
# (a pasta que contém src/) e passa a trabalhar a partir dela.
raiz = Path.cwd()
while not (raiz / "src").is_dir():
    if raiz.parent == raiz:
        raise FileNotFoundError("Pasta src não encontrada. Abra o notebook dentro do repositório.")
    raiz = raiz.parent
os.chdir(raiz)
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.leitura import ler_ano
from src.indicadores import NOMES, carregar_base, calcular_indicadores
from src.desenhos import NOVAS, PRIMEIRO_EXERCICIO_NOVAS, UFNT, montar_todos, resumo_registros, selecionar
from src.verificacoes import resumo_por_ano, verificar
from src.resumo import gravar_resumo

PASTA_ETAPA = Path("02_indicadores")
PASTA_TABELAS = PASTA_ETAPA / "tabelas"
PASTA_V2 = Path("versoes_anteriores/Versao_2/arquivos_tabulares")
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print(f"Raiz do repositório: {raiz}")
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")

## 1. Teste de regressão contra a Versão 2

Lê os dados brutos da Versão 2, calcula os 20 indicadores com as regras da Versão 2 (`regras="v2"`) e compara com os indicadores gravados pela Versão 2. A diferença máxima precisa ficar perto de zero (erro de arredondamento) em todos os indicadores. Se não ficar, o código mudou algo que não deveria.

As linhas sem movimento da extração antiga (criadas pela métrica de variação cambial) entram com valor zero, como na Versão 2.

In [ ]:
inicio = time.time()
mov_v2 = ler_ano(PASTA_V2 / "Exportação_completa_mestrado.zip")
mov_v2["mov"] = mov_v2["mov"].fillna(0.0)
sem_saldos = pd.DataFrame(columns=["ano", "mes", "cod_orgao", "cod_conta", "saldo"])  # as regras v2 não usam saldos
calculado = calcular_indicadores(mov_v2, sem_saldos, [2023, 2024, 2025], regras="v2").set_index("cod_orgao")
versao_2 = pd.read_csv(PASTA_V2 / "df_features_consolidadas.csv", sep=";", decimal=",").set_index("Cod_Orgao")

diferenca = (calculado[NOMES] - versao_2.loc[calculado.index, NOMES]).abs().max().rename("diferenca_maxima")
display(diferenca.to_frame().T)
assert set(calculado.index) == set(versao_2.index), "As universidades não batem com a Versão 2"
assert diferenca.max() < 1e-9, "Algum indicador não reproduz a Versão 2"
print(f"OK: os 20 indicadores das {len(calculado)} universidades reproduzem a Versão 2 "
      f"(diferença máxima de {diferenca.max():.1e}). Tempo: {time.time() - inicio:.0f} s")
del mov_v2

## 2. Tabelas dos desenhos

Cada tabela traz todos os registros possíveis do desenho. A coluna `motivo_fora` diz por que um registro fica fora da análise principal (vazia quando ele entra). Contagens esperadas na análise principal, conforme o diário: A 44/39, B 330/310, C 284/269 e D 369/349.

In [ ]:
inicio = time.time()
mov, sal = carregar_base()
tabelas = montar_todos(mov, sal)
for desenho, tabela in tabelas.items():
    tabela.to_parquet(PASTA_TABELAS / f"desenho_{desenho}.parquet", index=False)
with pd.ExcelWriter(PASTA_ETAPA / "tabelas_desenhos.xlsx", engine="openpyxl") as escritor:
    for desenho, tabela in tabelas.items():
        tabela.to_excel(escritor, sheet_name=f"desenho_{desenho}", index=False)
print(f"Tabelas gravadas em {PASTA_TABELAS}. Tempo: {time.time() - inicio:.0f} s")

registros = resumo_registros(tabelas)
display(registros)
esperado = {"A": (44, 39), "B": (330, 310), "C": (284, 269), "D": (369, 349)}
for _, linha in registros.iterrows():
    assert (linha["principal_44"], linha["principal_39"]) == esperado[linha["desenho"]], f"Contagem diferente no desenho {linha['desenho']}"
print("OK: contagens iguais às do diário.")

### Registros fora da análise principal, por motivo

In [ ]:
fora = pd.concat([t[t["motivo_fora"] != ""] for t in tabelas.values()])
display(fora.groupby(["desenho", "motivo_fora"]).size().rename("registros").reset_index())

principal = {d: selecionar(t, universo=44) for d, t in tabelas.items()}
vazios = {d: int(p.drop(columns=["motivo_fora"]).isna().sum().sum()) for d, p in principal.items()}
print("Valores vazios na análise principal (precisa ser zero em todos):", vazios)

## 3. Verificações simples do painel

Cinco verificações com regra contábil objetiva (D46), feitas fora dos modelos. A tabela mostra quantas universidades têm alerta em cada ano, sem a UFNT e sem o primeiro exercício das universidades novas. V2 e V5 são pontos de atenção, e não falhas (D48 e D46).

In [ ]:
verificacoes = verificar(mov, sal, anos=range(2017, 2026))
verificacoes.to_parquet(PASTA_TABELAS / "verificacoes.parquet", index=False)
universo = verificacoes[(verificacoes["cod_orgao"] != UFNT)
                        & ~(verificacoes["cod_orgao"].isin(NOVAS) & (verificacoes["ano"] <= PRIMEIRO_EXERCICIO_NOVAS))]
display(resumo_por_ano(universo))

## 4. Resumo dos indicadores

Grava `resumo_indicadores.xlsx` com as estatísticas de cada indicador na análise principal de cada desenho. As duas tabelas abaixo antecipam as regras da etapa 3: indicadores em que mais de 75% dos registros têm o mesmo valor e pares de indicadores com correlação de Spearman acima de 0,75 (em módulo). Os limites definitivos serão fixados no protocolo.

In [ ]:
resumo = gravar_resumo(PASTA_ETAPA / "resumo_indicadores.xlsx", tabelas, verificacoes)

mesmo_valor = pd.concat([resumo[f"estatisticas_{d}"].assign(desenho=d) for d in tabelas])
print("Indicadores com mais de 75% dos registros no mesmo valor:")
display(mesmo_valor.loc[mesmo_valor["pct_valor_mais_comum"] > 75,
                        ["desenho", "indicador", "valor_mais_comum", "pct_valor_mais_comum"]].round(2))

pares = pd.concat([resumo[f"pares_{d}"].assign(desenho=d) for d in tabelas])
print("Pares de indicadores com correlação acima de 0,75 (em módulo):")
display(pares[pares["acima_de_0_75"]].round(3))

## Registro

Preencher depois de rodar, para o diário de decisões:

- Data da execução:
- Teste de regressão:
- Contagens dos desenhos:
- Observações: